In [1]:
import pandas as pd
import glob
import os

# Path to the directory containing the CSV files
data_dir = "../../resources/data/costs/default/"
csv_files = glob.glob(os.path.join(data_dir, "costs_*.csv"))
csv_files = sorted(csv_files)
csv_files


['../../resources/data/costs/default/costs_2020.csv',
 '../../resources/data/costs/default/costs_2025.csv',
 '../../resources/data/costs/default/costs_2030.csv',
 '../../resources/data/costs/default/costs_2035.csv',
 '../../resources/data/costs/default/costs_2040.csv',
 '../../resources/data/costs/default/costs_2045.csv',
 '../../resources/data/costs/default/costs_2050.csv',
 '../../resources/data/costs/default/costs_2055.csv',
 '../../resources/data/costs/default/costs_2060.csv']

In [2]:
import os
import pandas as pd

# === 自定义要修改的参数 ===
target_tech = "nuclear"              # 要修改的技术名
target_param = "investment"       # 要修改的参数名
new_value = 3000                # 新的投资成本值（示例：500000 EUR/MW）

# === 遍历所有 CSV 文件 ===
for file_path in csv_files:
    basename = os.path.basename(file_path)
    year = int(basename.split("_")[1].split(".")[0])  # 假设文件名含年份，如 cost_2030.csv
    print(f"Processing {basename}...")

    # 读取 CSV
    df = pd.read_csv(file_path)

    # 筛选并修改目标行
    mask = (df["technology"] == target_tech) & (df["parameter"] == target_param)
    if mask.any():
        old_values = df.loc[mask, "value"].tolist()
        df.loc[mask, "value"] = new_value
        print(f"  ✅ Modified {target_tech} {target_param}: {old_values} → {new_value}")
    else:
        print(f"  ⚠️  No {target_tech} {target_param} found in {basename}")

    # === 直接覆盖保存 ===
    df.to_csv(file_path, index=False)
    print(f"  💾 Saved updated file to {file_path}\n")


Processing costs_2020.csv...
  ✅ Modified nuclear investment: [6000.0] → 3000
  💾 Saved updated file to ../../resources/data/costs/default/costs_2020.csv

Processing costs_2025.csv...
  ✅ Modified nuclear investment: [6000.0] → 3000
  💾 Saved updated file to ../../resources/data/costs/default/costs_2025.csv

Processing costs_2030.csv...
  ✅ Modified nuclear investment: [6000.0] → 3000
  💾 Saved updated file to ../../resources/data/costs/default/costs_2030.csv

Processing costs_2035.csv...
  ✅ Modified nuclear investment: [6000.0] → 3000
  💾 Saved updated file to ../../resources/data/costs/default/costs_2035.csv

Processing costs_2040.csv...
  ✅ Modified nuclear investment: [6000.0] → 3000
  💾 Saved updated file to ../../resources/data/costs/default/costs_2040.csv

Processing costs_2045.csv...
  ✅ Modified nuclear investment: [6000.0] → 3000
  💾 Saved updated file to ../../resources/data/costs/default/costs_2045.csv

Processing costs_2050.csv...
  ✅ Modified nuclear investment: [6000.0]

In [4]:
import pandas as pd
import glob
import os
import numpy as np

# -----------------------------
# 0. 设置人民币→欧元汇率 (可修改)
# -----------------------------
CNY_TO_EUR = 0.122     # 2013 年平均汇率

# -----------------------------
# 1. 要插值的年份
# -----------------------------
ALL_YEARS = [2020, 2025, 2030, 2035, 2040, 2045, 2050, 2055, 2060]

# -----------------------------
# 2. 加载文献表
# -----------------------------
lit_path = "/p/tmp/yanleizh/newpypsa_0913/PyPSA-China-PIK/resources/data/costs/reference_costs/tech_costs_subset_litreview.csv"
lit_df = pd.read_csv(lit_path)

solar_util = lit_df[(lit_df["technology"] == "solar-utility") &
                    (lit_df["cost_type"] == "investment")]

print("📌 可选文献来源：\n")
for i, row in solar_util.iterrows():
    print(f"[{i}] {row['unit']:10s} | {row['reference']} | {row['link']}")

choice = int(input("\n请输入你想使用的文献编号: "))
row = solar_util.loc[choice]

print("\n➡️ 你选择的是：")
print(row[["reference", "unit", "link"]], "\n")

# -----------------------------
# 3. 抽取原始年份
# -----------------------------
year_cols = [c for c in row.index if c.isdigit()]
available_years = [int(y) for y in year_cols]
available_values = row[year_cols].astype(float).values

# -----------------------------
# 4. 如有需要，人民币→欧元
# -----------------------------
unit = row["unit"].lower()
if "cny" in unit:
    print("💱 文献为人民币 → 自动转换为欧元")
    available_values = available_values * CNY_TO_EUR
elif "eur" in unit:
    print("💶 文献本身就是欧元，无需转换")
else:
    print("⚠️ 未识别单位，将按原值处理")

# -----------------------------
# 5. 插值所有中间年份
# -----------------------------
interp_values = np.interp(ALL_YEARS, available_years, available_values)
interp_series = pd.Series(interp_values, index=ALL_YEARS)

print("\n📌 插值后的最终欧元成本：")
print(interp_series, "\n")

# -----------------------------
# 6. 写入 default 成本文件 (使用 technology 作为列名)
# -----------------------------
default_dir = "../../resources/data/costs/default/"
csv_files = sorted(glob.glob(os.path.join(default_dir, "costs_*.csv")))

for f in csv_files:
    df = pd.read_csv(f)

    # ⚠️ 替换成你的真实列名
    mask = (df["technology"] == "solar") & (df["parameter"] == "investment")

    if mask.sum() == 0:
        print(f"⚠️ {os.path.basename(f)}: 未找到 technology=solar & parameter=investment，跳过")
        continue

    for year in ALL_YEARS:
        col = str(year)
        if col in df.columns:
            df.loc[mask, col] = interp_series[year]

    df.to_csv(f, index=False)
    print(f"✅ 已写入: {os.path.basename(f)}")

print("\n🎉 完成：solar 成本（含插值 + 欧元转换）已写入所有 default 成本文件！")


📌 可选文献来源：

[0] cny/kw     | Zhu et al. | Zhu Z, Zhang D, Zhang X, Zhang X. Integrated modeling for the transition pathway of China��s power system. Energy Environ Sci 2025:10.1039.D5EE00355E. https://doi.org/10.1039/D5EE00357E.
[1] $/mw       | Switch-China | https://github.com/switch-model/switch-china-open-model/
[2] cny/kw     | Sun et al. | Qixing Sun, Chao Zhang, Chengren Li, Peipei You, Xiao Gao, Qian Zhao, Zhao Xu, Sijia Liu, and Li Yanlin. Prediction of Power System Cost and Price Level Under the Goal of Carbon Peak and Carbon Neutralization�� (in Chinese).
[3] eur/kw     | PyPSA-China | nan
[4] cny/kw     | NREL ATB | https://atb.nrel.gov/electricity/2026/index
[5] cny/kw     | Li et al. | Li, Mingquan, Rui Shan, Ahmed Abdulla, Edgar Virguez, and Shuo Gao. "The role of dispatchability in China's power system decarbonization."?Energy & Environmental Science?17, no. 6 (2024): 2193-2207.

➡️ 你选择的是：
reference                                       NREL ATB
unit                     

In [ ]:
heat_rate = 300 # gce/kwh
gce_to_kwh = 8.14/1000
eta= 1/(gce_to_kwh*heat_rate)
eta

In [ ]:
heat_rate = 280 # gce/kwh
gce_to_kwh = 8.14/1000
eta= 1/(gce_to_kwh*heat_rate)
eta

In [ ]:
import numpy as np
# from heat rate coal eq/kWh (Xiang et al nat energy 2023) in 2020 to 52% 2060 (D KEA)
l = np.linspace(eta,0.52, len(csv_files))
years = [int(f.split('_')[-1].split('.csv')[0]) for f in csv_files]
new_values = dict(zip(years, l))
new_values

In [ ]:

for file_path in csv_files:
    # Extract year from filename
    basename = os.path.basename(file_path)
    year = basename.split("_")[1].split(".")[0]

    # Read CSV
    df = pd.read_csv(file_path)

    # Find the row for 'coal' and column for 'efficiency'
    mask = (df['technology'] == 'central coal CHP') & (df['parameter'] == 'efficiency')
    if mask.any():
        # Update the value to the new format: {year}:value
        old_value = df.loc[mask, 'value'].iloc[0]
        df.loc[mask, 'value'] = new_values[int(year)]*0.999
        df.loc[mask, "source"] = "Linear increase from Xiang et al nat energy 2023 to DKEA catalogue 2023 value for 2060"

        # # Overwrite the CSV file
        df.to_csv(file_path, index=False)

for file_path in csv_files:
    # Extract year from filename
    basename = os.path.basename(file_path)
    year = basename.split("_")[1].split(".")[0]

    # Read CSV
    df = pd.read_csv(file_path)

    # Find the row for 'coal' and column for 'efficiency'
    mask = (df['technology'] == 'coal') & (df['parameter'] == 'efficiency')
    if mask.any():
        # Update the value to the new format: {year}:value
        old_value = df.loc[mask, 'value'].iloc[0]
        df.loc[mask, 'value'] = new_values[int(year)]*1.001
        df.loc[mask, "source"] = "Linear increase from Xiang et al nat energy 2023 to DKEA catalogue 2023 value for 2060"

        # # Overwrite the CSV file
        df.to_csv(file_path, index=False)


In [ ]:

for file_path in csv_files:
    # Extract year from filename
    basename = os.path.basename(file_path)
    year = basename.split("_")[1].split(".")[0]

    # Read CSV
    df = pd.read_csv(file_path)

    # Find the row for 'coal' and column for 'efficiency'
    mask = (df['technology'] == 'central hydrogen CHP') & (df['parameter'] == 'efficiency')
    mask_gas = (df['technology'] == 'central gas CHP CC') & (df['parameter'] == 'efficiency')

    if mask.any():
        # Update the value to the new format: {year}:value
        old_value = df.loc[mask, 'value'].iloc[0]
        df.loc[mask, 'value'] = df.loc[mask_gas, 'value'].values*0.95
        df.loc[mask, "source"] = "gas CHP combined cycle efficiency *0.95 (based on a 60% H2 TEA https://www.mdpi.com/2071-1050/17/8/3369)"

        # # Overwrite the CSV file
        df.to_csv(file_path, index=False)

    # Read CSV
    df = pd.read_csv(file_path)

    # Find the row for 'coal' and column for 'efficiency'
    mask = (df['technology'] == 'central hydrogen CHP') & (df['parameter'] == 'investment')
    mask_gas = (df['technology'] == 'central gas CHP CC') & (df['parameter'] == 'investment')

    if mask.any():
        # Update the value to the new format: {year}:value
        old_value = df.loc[mask, 'value'].iloc[0]
        df.loc[mask, 'value'] = df.loc[mask_gas, 'value'].values*1.1
        df.loc[mask, "source"] = "10% markup on gas CC as per H2-Ready Gas-fired Power Plants, Christidis et al 2023, Reiner Lemoine Institut"

        # # Overwrite the CSV file
        df.to_csv(file_path, index=False)


    # Read CSV
    df = pd.read_csv(file_path)

    # Find the row for 'coal' and column for 'efficiency'
    mask = (df['technology'] == 'central hydrogen CHP') & (df['parameter'] == 'lifetime')
    mask_gas = (df['technology'] == 'central gas CHP CC') & (df['parameter'] == 'lifetime')

    if mask.any():
        # Update the value to the new format: {year}:value
        df.loc[mask, 'value'] = df.loc[mask_gas, 'value'].values-5
        df.loc[mask, "source"] = "5 year lifetime penalty vs gas - assumption"

        # # Overwrite the CSV file
        df.to_csv(file_path, index=False)



In [ ]:
eta_coal_boiler = 0.78 # efficiency guess of coal historical boilers *not suitable for pathway mode*
for file_path in csv_files:
    # Extract year from filename
    basename = os.path.basename(file_path)
    year = int(basename.split("_")[1].split(".")[0])
    
    # Read CSV
    df = pd.read_csv(file_path)
    df_ = pd.concat(
        [df, 
        pd.Series(["central coal boiler",year,"hist_efficiency",eta_coal_boiler,"p.u.", "expert guess for brownfield", "OPOP H4EKO class C"], index=df.columns).to_frame().T,
        pd.Series(["decentral coal boiler",year,"hist_efficiency",eta_coal_boiler,"p.u.", "expert guess for brownfield", "OPOP H4EKO class C"], index=df.columns).to_frame().T
        ])
    df_ = df_.drop_duplicates(subset=["technology", "parameter"], keep="first")
    df_.sort_values(by=["technology","year"], inplace=True)
    # # Overwrite the CSV file
    df_.to_csv(file_path, index=False)
    print("saved to ", file_path)


In [ ]:
df_.loc[df_.duplicated(subset=["technology", "parameter"], keep=False)]

In [ ]:
data = {}
for file_path in csv_files:
    # Extract year from filename
    basename = os.path.basename(file_path)
    year = basename.split("_")[1].split(".")[0]

    # Read CSV
    df = pd.read_csv(file_path)
    # Find the row for 'coal' and column for 'efficiency'
    mask = (df['technology'] == 'central gas CHP')
    if mask.any():
        # Update the value to the new format: {year}:value
        data[year] = df.loc[mask]
pd.concat(data.values()).query("parameter=='investment'").plot(kind="scatter", x="year", y="value")
pd.concat(data.values()).query("parameter=='efficiency'").plot(kind="scatter", x="year", y="value")

# FIX CC CHP

In [ ]:
# piecewise linear interpolation of DKEA values
dk_ea_invest = {
 2020: 0.88*1000,
 2030: 0.83*1000,
 2050: 0.8*1000}


In [ ]:
from scipy.optimize import curve_fit
import numpy as np
import matplotlib.pyplot as plt

# Prepare data
x = np.array(list(dk_ea_invest.keys()))
y = np.array(list(dk_ea_invest.values()))

# Exponential decay function
def exp_decrease(x, a, b, c):
    return a * np.exp(b * (x - x[0])) + c

# Fit
popt, _ = curve_fit(exp_decrease, x, y, p0=(y[0], -0.1, 750))

# popt contains the fitted parameters a and b
a, b, c = popt
print(f"Fitted parameters: a={a}, b={b}")

# Example: predict for all years
years_fit = np.arange(2020, 2061, 5)
y_fit = exp_decrease(years_fit, a, b, c)

plt.scatter(dk_ea.keys(), dk_ea.values(), marker="D", color = "black")
plt.scatter(years_fit, y_fit, color='red', alpha=0.9)
fit = dict(zip(years_fit, y_fit))

In [ ]:
data = {}
for file_path in csv_files:
    # Extract year from filename
    basename = os.path.basename(file_path)
    year = basename.split("_")[1].split(".")[0]

    # Read CSV
    df = pd.read_csv(file_path)
    # Find the row for 'coal' and column for 'efficiency'
    mask = (df['technology'] == 'central gas CHP CC') & (df['parameter'] == 'investment')
    if mask.any():
        # Update the value to the new format: {year}:value
        # Find the row for 'coal' and column for 'efficiency'
        if mask.any():
            # Update the value to the new format: {year}:value
            old_value = df.loc[mask, 'value'].iloc[0]
            df.loc[mask, 'value'] = f"{fit[int(year)]:.2f}"
            df.loc[mask,"further description"] = "05 Gas turb. CC, steam extract.: Investment"

            # # Overwrite the CSV file
            df.to_csv(file_path, index=False)

In [ ]:
# piecewise linear interpolation of DKEA values
dk_ea = {
 2020: 0.59,
 2030: 0.61,
 2050: 0.63}
years = [2020, 2025, 2030, 2035, 2040, 2045, 2050, 2055, 2060   ]
plt.plot(dk_ea.keys(), dk_ea.values(), marker="D", color = "black")
eff_interp = np.linspace(0.59,0.63,len(years)-2).tolist() + [0.63,0.63]
plt.plot(years, eff_interp, marker="o", color = "red")
eff_d = dict(zip(years, eff_interp))

In [ ]:
data = {}
for file_path in csv_files:
    # Extract year from filename
    basename = os.path.basename(file_path)
    year = basename.split("_")[1].split(".")[0]

    # Read CSV
    df = pd.read_csv(file_path)
    # Find the row for 'coal' and column for 'efficiency'
    mask = (df['technology'] == 'central gas CHP CC') & (df['parameter'] == 'efficiency')
    if mask.any():
        # Update the value to the new format: {year}:value
        # Find the row for 'coal' and column for 'efficiency'
        if mask.any():
            # Update the value to the new format: {year}:value
            old_value = df.loc[mask, 'value'].iloc[0]
            df.loc[mask, 'value'] = eff_d[int(year)]
            df.loc[mask,"further description"] = "05 Gas turb. CC, steam extract.:  Efficiency"

            # # Overwrite the CSV file
            df.to_csv(file_path, index=False)